In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.multiclass import OneVsRestClassifier

In [ ]:
num_iter = 10
split_folder = "neuron_data_splits"
save_name = "neuron_classification_results.csv"
importances_save_name = "neuron_feature_importances.csv"

# Feature Names 
df_name_features = [
    'N_edg', 'Betw', 'Spec_ent', 'Alg_Conn', 'Assort', 'G_diam',
    'Br_dens', 'Br_l', 'Energy', 'Apl', 'Circ', 'FD',
    'Dir_std', 'q1', 'q2', 'q3', 'q4', 'Dir_ent', 'Order'
]

# Region mapping for reporting
regions = ['Basal Ganglia', 'Cerebellum', 'Hippocampus', 'Olfactory Bulb', 'Retina']
region_keys = ['bg', 'cb', 'hp', 'mob', 'rt']

# Storage for results
neuron_results = [["Classifier", "Accuracy on Test Set", "Std"]]
results = [neuron_results]

# Initialize storage for each region + overall
location_accuracies = {key: [] for key in region_keys + ['overall']}
all_importances = [] 



for i in range(num_iter):
    print(f"\n--- Processing Split {i} ---")
    
    # 1. Load the pre-saved split
    try:
        train_df = pd.read_csv(f"{split_folder}/train_split_{i}.csv")
        test_df = pd.read_csv(f"{split_folder}/test_split_{i}.csv")
    except FileNotFoundError:
        print(f"Error: Could not find split files in '{split_folder}'. Run 'generate_neuron_splits.py' first.")
        break
    
    # Separate Features (X) and Labels (y)
    y_train = train_df['label_target'].values
    X_train = train_df.drop('label_target', axis=1).values
    
    y_test = test_df['label_target'].values
    X_test = test_df.drop('label_target', axis=1).values
    
    # Normalize (Fit on TRAIN, transform TEST)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Train Random Forest (OneVsRest)
    rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=20, criterion='gini', random_state=0)
    clf = OneVsRestClassifier(rf_classifier)
    clf.fit(X_train, y_train)
    
    # Extract Feature Importance
    importances_list = []
    for estimator in clf.estimators_:
        importances_list.append(estimator.feature_importances_)
    iter_mean_importances = np.mean(np.array(importances_list), axis=0) * 100
    all_importances.append(iter_mean_importances)

    # Test and Evaluate
    pred_test = clf.predict(X_test)
    score = accuracy_score(y_test, pred_test)
    print(f"Accuracy: {score:.4f}")
    

    print(f"Classification report for Split {i}")
    print(classification_report(y_test, pred_test, target_names=regions, zero_division=0))

    print(f"Confusion Matrix for Split {i}:")
    print(confusion_matrix(y_test, pred_test))
    
    location_accuracies['overall'].append(score)
    
    # Per-class accuracy calculation
    # 0=BG, 1=CB, 2=HP, 3=MOB, 4=RT
    for idx, key in enumerate(region_keys):
        mask = (y_test == idx)
        if np.sum(mask) > 0:
            loc_acc = accuracy_score(y_test[mask], pred_test[mask])
        else:
            loc_acc = 0
        location_accuracies[key].append(loc_acc)


# FINAL ACCURACY SUMMARY

mean_perf = np.mean(location_accuracies['overall'])
std_perf = np.std(location_accuracies['overall'])
results[0].append(["RandomForest", mean_perf, std_perf])

print("\n" + "="*40)
print("FINAL RESULTS (Mean ± Std over 10 splits)")
print("="*40)
location_summary = {}

# Print Headers
headers = " | ".join([key.upper() for key in region_keys + ['overall']])
print(headers)

# Format Data
for key in region_keys + ['overall']:
    mean_acc = np.mean(location_accuracies[key])
    std_acc = np.std(location_accuracies[key])
    # Convert to percentage logic for display if desired, here kept as ratio 0-1
    location_summary[key] = f"{mean_acc:.4f} ± {std_acc:.4f}"

# Print Values
print(" | ".join([location_summary[key] for key in region_keys + ['overall']]))

# Save Results
results_df = pd.DataFrame(results[0][1:], columns=results[0][0])
results_df.to_csv(save_name, index=False)
print(f"\nAccuracy results saved to: {save_name}")


# FEATURE IMPORTANCE SUMMARY & PLOT

all_importances = np.array(all_importances) 

final_mean_importances = np.mean(all_importances, axis=0)
final_std_importances = np.std(all_importances, axis=0)

sorted_indices = np.argsort(final_mean_importances)[::-1]
sorted_feature_names = np.array(df_name_features)[sorted_indices]
sorted_mean_importances = final_mean_importances[sorted_indices]
sorted_std_importances = final_std_importances[sorted_indices]

print("\nAggregated Feature Importances:")
for name, imp, std in zip(sorted_feature_names, sorted_mean_importances, sorted_std_importances):
    print(f"{name}: {imp:.2f}% ± {std:.2f}")

# Save Feature Importances
importances_df = pd.DataFrame({
    'Feature': sorted_feature_names,
    'Mean Importance (%)': np.round(sorted_mean_importances, 2),
    'Std Importance': np.round(sorted_std_importances, 2)
})
importances_df.to_csv(importances_save_name, index=False)

# PLOT
plot_feature_names = sorted_feature_names[::-1]
plot_mean_importances = sorted_mean_importances[::-1]
plot_std_importances = sorted_std_importances[::-1]

fig, ax = plt.subplots(figsize=(12, 8))
forest_importances = pd.Series(plot_mean_importances, index=plot_feature_names)
forest_importances.plot.barh(ax=ax, xerr=plot_std_importances, color='teal', edgecolor='black')

ax.set_title("Aggregated Feature Importances (Neuron Data)", fontsize=18)
ax.set_xlabel("Mean Decrease in Impurity (%)", fontsize=14)
sns.despine()
plt.tight_layout()
plt.show()